# Mediterranean Sea Outflow Plume

East-west salinity and temperature transects through the Mediterranean Sea
outflow plume near 36°N, comparing model output against WOA or Argo
observations.

Authors: John Krasting

In [ ]:
# Development mode: constantly refreshes module code
%load_ext autoreload
%autoreload 2

## Framework Code and Diagnostic Setup

In [ ]:
import esnb
from esnb import NotebookDiagnostic, RequestedVariable, CaseGroup2, nbtools
from esnb.sites.gfdl import call_dmget, convert_to_momgrid
import climplot

In [ ]:
# Define requested variables: salinity and temperature on annual z-space grid
variables = [
    RequestedVariable("so", "ocean_annual_z", frequency="yearly"),
    RequestedVariable("thetao", "ocean_annual_z", frequency="yearly"),
]

# Diagnostic configuration
diag_name = "Mediterranean Outflow"
diag_desc = "Salinity and temperature transects through the Med outflow plume"
user_options = {"observations": "WOA"}

# Initialize the diagnostic
diag = NotebookDiagnostic(diag_name, diag_desc, variables=variables, **user_options)

In [ ]:
# Define experiments to analyze
groups = [
    CaseGroup2("odiv-516", date_range=("1993-01-01", "2017-12-31"), name="OM5 B11 NB"),
    CaseGroup2("odiv-290", date_range=("1993-01-01", "2017-12-31"), name="OM5 B01 (OM4-like)"),
]

In [ ]:
# Combine experiments with the diagnostic request and determine files to load
diag.resolve(groups)

In [ ]:
# Print a list of file paths
_ = [print(x) for x in diag.files]

*(The files above are necessary to run the diagnostic.)*

In [ ]:
# Stage files from mass storage
call_dmget(diag.files, status=True)
call_dmget(diag.files)

In [ ]:
# Load the data as xarray datasets
diag.open()

In [ ]:
convert_to_momgrid(diag, return_corners=True)

## Main Diagnostic

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import cmocean
import momgrid

os.environ["MOMGRID_WEIGHTS_DIR"] = "/nbhome/jpk/grid_weights"

In [ ]:
all_figs = []
climplot.publication()

In [ ]:
# Load observations based on user option
obs_source = diag.diag_vars.get("observations", "WOA")

if obs_source == "WOA":
    obs_label = "WOA23"
    ds_t_obs = xr.open_dataset(
        "/net3/bgr/Datasets/WOA/woa23_decav91C0_t00_04.nc",
        decode_times=False,
    )
    ds_s_obs = xr.open_dataset(
        "/net3/bgr/Datasets/WOA/woa23_decav91C0_s00_04.nc",
        decode_times=False,
    )
    # WOA uses 0-360 longitudes; convert to -180 to 180
    ds_t_obs = ds_t_obs.assign_coords(lon=(ds_t_obs.lon + 180) % 360 - 180)
    ds_t_obs = ds_t_obs.sortby("lon")
    ds_s_obs = ds_s_obs.assign_coords(lon=(ds_s_obs.lon + 180) % 360 - 180)
    ds_s_obs = ds_s_obs.sortby("lon")

    # Select transect domain and nearest latitude to 36N
    obs_t = ds_t_obs.t_an.sel(
        lon=slice(-75, 15),
    ).sel(lat=36, method="nearest").squeeze()
    obs_s = ds_s_obs.s_an.sel(
        lon=slice(-75, 15),
    ).sel(lat=36, method="nearest").squeeze()
    obs_lon = obs_t.lon.values
    obs_depth = obs_t.depth.values

elif obs_source == "Argo":
    obs_label = "RG Argo"
    ds_argo = xr.open_mfdataset(
        "/net2/bgr/Data/ARGO/RG_Clim/Updated/RG_ArgoClim_*.nc",
        decode_times=False,
    )
    ds_argo = ds_argo.assign_coords(
        LONGITUDE=(ds_argo.LONGITUDE + 180) % 360 - 180,
    )
    ds_argo = ds_argo.sortby("LONGITUDE")
    obs_t = ds_argo.ARGO_TEMPERATURE_MEAN.sel(
        LONGITUDE=slice(-75, 15),
    ).sel(LATITUDE=36, method="nearest").squeeze().mean("TIME")
    obs_s = ds_argo.ARGO_SALINITY_MEAN.sel(
        LONGITUDE=slice(-75, 15),
    ).sel(LATITUDE=36, method="nearest").squeeze().mean("TIME")
    obs_lon = obs_t.LONGITUDE.values
    obs_depth = obs_t.PRESSURE.values

else:
    raise ValueError(f"Unknown observations source: {obs_source}")

# Infer cell-edge coordinates for pcolormesh
obs_z_i = momgrid.util.infer_bounds(obs_depth)
obs_lon_i = momgrid.util.infer_bounds(obs_lon)

obs_data = {
    "thetao": obs_t.values,
    "so": obs_s.values,
    "lon": obs_lon,
    "depth": obs_depth,
    "z_i": obs_z_i,
    "lon_i": obs_lon_i,
}

In [ ]:
# Transect configuration
transect = {
    "lat": 36,
    "lon_lim": [-75, 15],
    "salt_levels": np.linspace(35, 36.6, 21),
    "temp_levels": np.linspace(5, 15, 11),
    "z_max": 5500,
}

In [ ]:
# Process each experiment: extract lon-depth transect
nexps = len(diag.groups)
model_data = {}

for group in diag.groups:
    ds_s = group.datasets[diag.varmap["so"]]
    ds_t = group.datasets[diag.varmap["thetao"]]

    # Time-average
    so_mean = ds_s.so.mean("time")
    thetao_mean = ds_t.thetao.mean("time")

    # Find nearest y-index to transect latitude
    lat_1d = ds_s.geolat.values[:, 0]
    ii = int(np.argmin(np.abs(lat_1d - transect["lat"])))

    # Extract transect along x-dimension at fixed y-index
    so_transect = so_mean.isel(yh=ii)
    thetao_transect = thetao_mean.isel(yh=ii)

    # Model longitude at transect (center coordinates)
    mod_lon = ds_s.geolon.values[ii, :]

    # Depth interfaces for pcolormesh
    if "z_i" in ds_s.coords:
        z_i = ds_s.z_i.values
    else:
        z_i = momgrid.util.infer_bounds(ds_s.z_l.values)

    # Bottom topography from momgrid static file
    model_type = ds_s.attrs.get("model", "")
    grid = momgrid.MOMgrid(model_type, warn=False).to_xarray()
    deptho = grid.deptho.values[ii, :]

    # Trim to longitude domain
    lon_mask = (mod_lon >= transect["lon_lim"][0]) & (
        mod_lon <= transect["lon_lim"][1]
    )

    # Infer longitude cell-edges for pcolormesh
    mod_lon_sub = mod_lon[lon_mask]
    mod_lon_i = momgrid.util.infer_bounds(mod_lon_sub)

    model_data[group] = {
        "so": so_transect.values[:, lon_mask],
        "thetao": thetao_transect.values[:, lon_mask],
        "lon": mod_lon_sub,
        "lon_i": mod_lon_i,
        "z_l": ds_s.z_l.values,
        "z_i": z_i,
        "deptho": deptho[lon_mask],
    }

In [ ]:
# Plot salinity transect: model panels + observation panel
fig, axes = plt.subplots(
    nexps + 1, 1, figsize=(10, 3 * (nexps + 1)), dpi=150,
)

cmap_s, norm_s, _ = climplot.sequential_cmap(
    transect["salt_levels"][0],
    transect["salt_levels"][-1],
    float(np.diff(transect["salt_levels"][:2])[0]),
    cmap_name="cmo.haline",
)

# Model panels
for n, group in enumerate(diag.groups):
    ax = axes[n]
    ax.set_facecolor("gray")
    d = model_data[group]

    cb = ax.pcolormesh(
        d["lon_i"], d["z_i"], d["so"],
        cmap=cmap_s, norm=norm_s,
    )

    # Overlay observation contours (red dashed)
    ax.contour(
        obs_data["lon"], obs_data["depth"], obs_data["so"],
        levels=transect["salt_levels"][::3],
        colors="red", linewidths=0.8, linestyles="dashed",
    )

    # Bottom topography
    ax.fill_between(d["lon"], transect["z_max"], d["deptho"], color="gray")

    ax.set_ylim(transect["z_max"], 0)
    ax.set_xlim(transect["lon_lim"])
    ax.set_ylabel("Depth [m]")
    ax.set_title(group.name)

# Observation panel
ax = axes[nexps]
ax.set_facecolor("#d0d0d0")
cb = ax.pcolormesh(
    obs_data["lon_i"], obs_data["z_i"], obs_data["so"],
    cmap=cmap_s, norm=norm_s,
)
ax.contour(
    obs_data["lon"], obs_data["depth"], obs_data["so"],
    levels=transect["salt_levels"][::3],
    colors="red", linewidths=0.8, linestyles="dashed",
)
ax.set_ylim(transect["z_max"], 0)
ax.set_xlim(transect["lon_lim"])
ax.set_ylabel("Depth [m]")
ax.set_xlabel("Longitude")
ax.set_title(f"{obs_label} Observations")

climplot.add_panel_labels(axes.flatten())

plt.subplots_adjust(bottom=0.10, hspace=0.35)
cax = fig.add_axes([0.15, 0.03, 0.70, 0.02])
fig.colorbar(
    cb, cax=cax, orientation="horizontal",
    label="Salinity [PSU]", extend="both",
)

all_figs.append(fig)

In [ ]:
# Plot temperature transect: model panels + observation panel
fig, axes = plt.subplots(
    nexps + 1, 1, figsize=(10, 3 * (nexps + 1)), dpi=150,
)

cmap_t, norm_t, _ = climplot.sequential_cmap(
    transect["temp_levels"][0],
    transect["temp_levels"][-1],
    float(np.diff(transect["temp_levels"][:2])[0]),
    cmap_name="cmo.thermal",
)

# Model panels
for n, group in enumerate(diag.groups):
    ax = axes[n]
    ax.set_facecolor("gray")
    d = model_data[group]

    cb = ax.pcolormesh(
        d["lon_i"], d["z_i"], d["thetao"],
        cmap=cmap_t, norm=norm_t,
    )

    # Overlay observation contours (blue dashed)
    ax.contour(
        obs_data["lon"], obs_data["depth"], obs_data["thetao"],
        levels=transect["temp_levels"][::3],
        colors="blue", linewidths=0.8, linestyles="dashed",
    )

    # Bottom topography
    ax.fill_between(d["lon"], transect["z_max"], d["deptho"], color="gray")

    ax.set_ylim(transect["z_max"], 0)
    ax.set_xlim(transect["lon_lim"])
    ax.set_ylabel("Depth [m]")
    ax.set_title(group.name)

# Observation panel
ax = axes[nexps]
ax.set_facecolor("#d0d0d0")
cb = ax.pcolormesh(
    obs_data["lon_i"], obs_data["z_i"], obs_data["thetao"],
    cmap=cmap_t, norm=norm_t,
)
ax.contour(
    obs_data["lon"], obs_data["depth"], obs_data["thetao"],
    levels=transect["temp_levels"][::3],
    colors="blue", linewidths=0.8, linestyles="dashed",
)
ax.set_ylim(transect["z_max"], 0)
ax.set_xlim(transect["lon_lim"])
ax.set_ylabel("Depth [m]")
ax.set_xlabel("Longitude")
ax.set_title(f"{obs_label} Observations")

climplot.add_panel_labels(axes.flatten())

plt.subplots_adjust(bottom=0.10, hspace=0.35)
cax = fig.add_axes([0.15, 0.03, 0.70, 0.02])
fig.colorbar(
    cb, cax=cax, orientation="horizontal",
    label="Temperature [°C]", extend="both",
)

all_figs.append(fig)

In [ ]:
# Register metrics for each experiment
for group in diag.groups:
    d = model_data[group]

    # Find index nearest to 1000m depth
    iz_1000 = int(np.argmin(np.abs(d["z_l"] - 1000)))

    # Max salinity at ~1000m along transect
    max_salt_1000 = float(np.nanmax(d["so"][iz_1000, :]))
    group.add_metric(
        "med_outflow_salt", ("max_at_1000m", round(max_salt_1000, 4)),
    )

    # Model-obs salinity difference at 10W, 1000m
    ix_10w = int(np.argmin(np.abs(d["lon"] - (-10))))
    model_s_10w = float(d["so"][iz_1000, ix_10w])

    obs_ix_10w = int(np.argmin(np.abs(obs_data["lon"] - (-10))))
    obs_iz_1000 = int(np.argmin(np.abs(obs_data["depth"] - 1000)))
    obs_s_10w = float(obs_data["so"][obs_iz_1000, obs_ix_10w])

    bias_10w = round(model_s_10w - obs_s_10w, 4)
    group.add_metric("med_outflow_salt", ("bias_10W_1000m", bias_10w))

### Write Metrics to File

In [ ]:
diag.write_metrics("med_outflow_metrics.json")

### Make a PowerPoint Presentation of Figures

In [ ]:
nbtools.save_pptx(all_figs, "med_outflow.pptx")